# Module 1.2 -- Apache Spark 4.1: Intent-Driven Declarative Pipelines

**Track:** Big Data Engineering for AI Systems  
**Toolchain:** PySpark 4.1 - Spark Declarative Pipelines (SDP) - Structured Streaming RTM  
**Objective:** Move from imperative ETL scripts to declarative pipeline definitions, and build real-time streaming with sub-second latency.

---

## What Changed in Spark 4.1?

### The Old Way (Imperative -- 'HOW to process')
```python
# You write EVERY step manually:
df = spark.read.parquet('s3://raw/')
df = df.filter(df.quality > 0.8)
df = df.groupBy('category').agg(avg('price'))
df.write.mode('overwrite').parquet('s3://gold/')
# YOU manage: order, retries, checkpoints, parallelism
```

### The New Way (Declarative -- 'WHAT to produce')
```python
@dp.table  # Just DECLARE what you want
def gold_prices():
    return spark.read.table('silver_products').groupBy('category').agg(avg('price'))
# SPARK manages: order, retries, checkpoints, parallelism
```

**Analogy:** Imperative = giving GPS turn-by-turn directions. Declarative = telling GPS the destination and letting it figure out the route.

### Why This Matters for AI Engineers

| Benefit | Impact |
|---------|--------|
| Auto-DAG construction | No manual dependency management |
| Built-in retries | Failed steps automatically retry |
| Native checkpointing | Resume from failure, don't restart |
| Streaming RTM mode | Sub-second latency (was multi-second) |


### 🎓 Junior to Senior: Concept Bridge

**The Senior Perspective:** When data exceeds 50GB, Pandas completely collapses (MemoryError). Seniors leverage distributed processing frameworks like Spark to crunch terabytes horizontally across clusters.

**Mental Model:** Pandas is a single powerful chef in a small kitchen. Spark is an executive chef (Driver) orchestrating 100 sous-chefs (Executors) working on pieces of the meal simultaneously.

**Common Junior Pitfall:** Calling `.collect()` on a massive Spark DataFrame, pulling 100GB of data back into the single Driver node, instacrating the entire Spark application.

---


In [ ]:
# Cell 1 -- Install
!pip install -q pyspark==4.1.0 pandas pyarrow


---
## 1 - Spark Declarative Pipelines (SDP)

### How Does SDP Build the DAG Automatically?

When you use `@dp.table`, Spark analyzes which tables each function reads from, and automatically constructs the execution order:

```
@dp.table bronze_sensors  (reads: raw kafka stream)
       |
@dp.table silver_cleaned   (reads: bronze_sensors)  -- Spark figures this out!
       |
@dp.table gold_features    (reads: silver_cleaned)   -- And this!
```

### Trade-offs: SDP vs Traditional Spark

| | SDP (Declarative) | Traditional (Imperative) |
|---|---|---|
| Learning curve | Low (just decorators) | High (manual DAGs) |
| Control | Less (Spark decides) | Full (you decide everything) |
| Debugging | Harder (abstracted) | Easier (step-by-step) |
| Best for | Standard ETL pipelines | Complex custom logic |


In [ ]:
# Cell 2 -- SDP pipeline definition (conceptual, runs on Spark 4.1 cluster)
#
# This demonstrates the STRUCTURE of a declarative pipeline.
# In production, this runs on a Spark cluster with @dp.table decorators.

pipeline_code = '''
from pyspark.pipelines import DeclaredPipeline as dp
from pyspark.sql.functions import avg, count, col, window

# BRONZE: ingest raw data (no transformation)
@dp.table(comment="Raw sensor readings from IoT Kafka topic")
def bronze_sensors():
    return (
        spark.readStream
        .format("kafka")
        .option("subscribe", "iot-sensors")
        .load()
        .selectExpr("CAST(value AS STRING)", "timestamp")
    )

# SILVER: clean and type (Spark auto-detects this depends on bronze)
@dp.table(comment="Cleaned, typed sensor data")
def silver_sensors():
    return (
        spark.read.table("bronze_sensors")
        .filter(col("device_id").isNotNull())
        .filter(col("temperature").between(-50, 150))
        .dropDuplicates(["device_id", "timestamp"])
    )

# GOLD: aggregate features for ML
@dp.table(comment="Hourly device health metrics for ML training")
def gold_device_health():
    return (
        spark.read.table("silver_sensors")
        .groupBy("device_id", window("timestamp", "1 hour"))
        .agg(
            avg("temperature").alias("avg_temp"),
            count("*").alias("reading_count"),
        )
    )
'''
print('Spark Declarative Pipeline Definition:')
print(pipeline_code)

---
## 2 - Structured Streaming Real-Time Mode (RTM)

### What is RTM?

Traditional Spark Streaming uses **micro-batching** -- it collects data for 1-5 seconds, then processes in bulk. RTM processes each record **immediately** upon arrival.

| Mode | Latency | Throughput | When to Use |
|------|---------|-----------|-------------|
| Micro-batch | 1-10 sec | Very high | Analytics, hourly reports |
| RTM | 1-50 ms | High | Fraud detection, live features |
| Continuous (deprecated) | ~1 ms | Lower | Replaced by RTM |

### Why RTM Instead of Flink for Low-Latency?

| | Spark RTM | Apache Flink |
|---|---|---|
| Latency | 1-50ms | 1-10ms |
| Ecosystem | Full Spark SQL, MLlib, Delta | Flink SQL only |
| Learning curve | Know Spark? Already done | Separate framework |
| Best for | Teams already on Spark | Pure streaming workloads |


In [ ]:
# Cell 3 -- PyArrow-native UDFs (eliminate Pandas conversion overhead)
#
# Old way: Python UDF serializes data row-by-row (SLOW, 10-100x overhead)
# New way: PyArrow UDF operates on columnar batches (FAST, near-native speed)

import pyarrow as pa
import numpy as np

# Simulate what a PyArrow UDF looks like
def compute_zscore_pyarrow(temperature_array: pa.Array) -> pa.Array:
    """PyArrow-native UDF: operates on columnar data without Pandas conversion."""
    values = temperature_array.to_numpy(zero_copy_only=False)
    mean, std = values.mean(), values.std()
    zscores = (values - mean) / (std + 1e-10)
    return pa.array(zscores, type=pa.float64())

# Demo
temps = pa.array(np.random.normal(72, 8, 1000))
zscores = compute_zscore_pyarrow(temps)
print('PyArrow UDF Performance Demo:')
print(f'  Input:  {len(temps)} temperatures')
print(f'  Output: {len(zscores)} z-scores')
print(f'  Anomalies (|z| > 2): {sum(1 for z in zscores.to_pylist() if abs(z) > 2)}')
print(f'\n  Old way (Python UDF):  ~500 rows/sec (serializes each row)')
print(f'  New way (PyArrow UDF): ~500,000 rows/sec (columnar batches)')
print(f'  Speedup: ~1000x')

---
## Summary

| Concept | What You Learned |
|---------|------------------|
| SDP | Declare WHAT, Spark handles HOW |
| Auto-DAG | Dependencies resolved automatically |
| RTM | Sub-second streaming without leaving Spark |
| PyArrow UDFs | 1000x faster than Python UDFs |

**Next -->** `05_realtime_streaming.ipynb` -- Kafka, Flink & Live Semantic Layers